# Drive -> SRT (ruso) con `faster-whisper-xxl` (large-v2)

Notebook autocontenido. Levanta videos desde tu Drive (carpeta **Host Videos**), los transcribe a SRT en ruso con el binario standalone **Faster-Whisper-XXL r245.4** de Purfview (large-v2 + extracción de voz Kim2), y guarda los `.srt` en tu Drive (carpeta **Subs_RU**).

**Antes de correr:** `Runtime -> Change runtime type -> T4 GPU`.

## Dos modos

- **Celda 1 — automático (coordinado por Google Sheet).** Las cuentas se reparten el trabajo solas usando un Sheet compartido como cola. Cada una toma la próxima fila `TODO` cuyo video tenga subido y la marca `DOING` mientras la procesa, `DONE` al terminar. Es **coherente entre cuentas al instante** (a diferencia de Drive), así que no hay colisiones. Recomendado para los 500+.
- **Celdas 2 + 3 — manual por rango en bloque.** Si querés bloques contiguos (`1-100`, `101-200`…): corré el setup (celda 2), elegí el rango y transcribí (celda 3).

Los dos modos usan los mismos parámetros de Whisper y guardan el SRT con el mismo nombre del video (`pepe.mp4` -> `pepe.srt`).

## Parámetros

```
-m large-v2                 -l ru                 --task transcribe
--initial_prompt None       --reprompt False
--condition_on_previous_text False
--hallucination_silence_threshold 4
--compute_type float16      --temperature 0       --beam_size 5
--vad_filter True
--ff_vocal_extract mdx_kim2 --voc_device cuda     --ff_loudnorm
-f srt                      --max_line_width 200  --max_line_count 1   --sentence
```

> **Limpieza de audio:** `--ff_vocal_extract mdx_kim2` separa la voz (MDX-Net Kim2) en la misma GPU antes de transcribir; la primera corrida del runtime baja el modelo Kim2 (cientos de MB) y sube el tiempo por video (~1.5x-2x).
> **Diarización opcional:** `--diarize pyannote_v3.1 --diarize_device cuda` están comentados en `build_cmd`; descomentalos si querés etiquetas de hablante.
> **Salida:** tu comando local usa `-o source` (SRT al lado del video). Acá va a `MyDrive/Subs_RU/<stem>.srt` vía carpeta temporal.


## 1) Modo automático — cola coordinada por Google Sheet (RECOMENDADO)

Corré **solo esta celda** en cada una de tus cuentas de Colab. La cola vive en un Google Sheet compartido (en `Subs_RU/Status_Videos`); cada Colab lee y escribe contra el servidor del Sheet, no contra Drive — eso es lo que permite coordinar entre cuentas sin colisiones.

**Mecánica por iteración:**

1. Lee toda la tabla del Sheet en 1 request.
2. Filtra: filas con `status=TODO` (o `DOING` con `claim_time` vencido por TTL) **cuyo video tenga subido en `Host Videos` ahora mismo**. Si no lo tiene local, la salteá — otra cuenta con ese archivo la va a agarrar.
3. Toma la primera. Marca `status=DOING`, `worker=<mi id>`, `claim_time=<ahora>`.
4. Espera 3s y relee esa fila. Si quedó mi worker, gané; si no, perdí la carrera y voy a la siguiente sin esperar.
5. Transcribe el video, deposita el SRT en `Subs_RU`, marca la fila `DONE`.
6. **Tolerancia a caídas:** una fila `DOING` con `claim_time` > 30 min sin pasar a `DONE` se considera huérfana — la próxima cuenta libre la re-claimea.
7. **Reconciliación automática:** si encuentra una fila no `DONE` cuyo SRT ya existe en Drive (porque la transcribiste antes con otro método), la marca `DONE` sin reprocesar.
8. **Espera y reintenta:** si no hay nada que pueda procesar ahora (todo lo TODO no está subido a esta cuenta), espera 60s y vuelve a chequear el Sheet — útil cuando vas subiendo tandas de a 100.

**Setup que tenés que tener listo una sola vez:**
- Un Google Sheet con la lista de los 758 nombres en la columna A (con encabezados `video | status | worker | claim_time` en la fila 1).
- El Sheet compartido como **editor** con las 8 cuentas (basta con dejarlo dentro de `Subs_RU`, que ya está compartido).
- El `SHEET_ID` (lo sacás de la URL del Sheet) pegado en la constante al principio de la celda.

**Para ver el progreso global:** abrí el Sheet en cualquier momento. Cuántas DONE, cuáles DOING y por qué cuenta, lo ves de una.


In [ ]:
!apt-get -qq install -y ffmpeg p7zip-full > /dev/null

import os, time, re, subprocess, shutil
from pathlib import Path
from google.colab import drive
import torch

# --- Descargar + extraer faster-whisper-xxl (una sola vez por sesion) ---
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"

if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, \
            f"Descarga fallo o quedo incompleta (size={ARCHIVE.stat().st_size if ARCHIVE.exists() else 0})"
    print("Extrayendo (puede tardar 1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), f"Extraccion fallo - no aparecio {EXE}"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)

# --- Mount Drive ---
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# --- Carpetas (editar si tus nombres son distintos) ---
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: 'float16' y la extraccion de voz en 'cuda' van a fallar. Activa T4 GPU.")

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    # Orden "humano": los tramos de digitos se comparan como enteros,
    # asi 2 < 10 < 100 (no lexicografico, donde "100" caeria antes que "2").
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS),
                key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"

total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
pending = total - already
print(f"\n{total} archivo(s) en '{INPUT_DIR.name}'  |  {already} con SRT  |  {pending} pendientes")
print("Primeros 5:")
for i, p in enumerate(inputs[:5], 1):
    mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
    print(f"  {i:>4}. [{mark}] {p.name}")
if total > 10:
    print("  ...")
    for i, p in enumerate(inputs[-3:], total - 2):
        mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
        print(f"  {i:>4}. [{mark}] {p.name}")

# --- Carpeta temporal + armador del comando (flags en UN solo lugar) ---
TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

def build_cmd(vid):
    return [
        str(EXE), str(vid),
        # --- Modelo / idioma ---
        "--model", "large-v2",
        "--language", "ru",
        "--task", "transcribe",
        # --- Decoder ---
        "--initial_prompt", "None",
        "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "float16",
        "--temperature", "0",
        "--beam_size", "5",
        "--vad_filter", "True",
        # --- Audio: limpieza para musica / ruido / multi-hablante ---
        "--ff_vocal_extract", "mdx_kim2",
        "--voc_device", "cuda",
        "--ff_loudnorm",
        # --- Diarizacion: descomenta SOLO si necesitas etiquetas de hablante (pasada extra pesada) ---
        # "--diarize", "pyannote_v3.1",
        # "--diarize_device", "cuda",
        # --- Salida ---
        "--max_line_width", "200",
        "--max_line_count", "1",
        "--sentence",
        "--output_dir", str(TMP_OUT),
        "--output_format", "srt",
    ]

def transcribe_one(vid, label):
    """Corre el CLI para un video y mueve el SRT a Drive (sin pisar uno ya hecho).
    Devuelve 'done' | 'skipped' | 'failed'."""
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except OSError: pass
    t0 = time.time()
    result = subprocess.run(build_cmd(vid), capture_output=True, text=True)
    cli_srt = TMP_OUT / f"{vid.stem}.srt"
    if result.returncode == 0 and cli_srt.exists():
        if final_srt.exists():                       # otro la dejo mientras transcribia
            cli_srt.unlink(missing_ok=True)
            print(f"   = {label}: ya estaba hecho por otra cuenta; descarto el mio")
            return "skipped"
        shutil.move(str(cli_srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"
    print(f"   x {label}: CLI exit {result.returncode}; sin SRT.")
    err = (result.stderr or "")[-400:]
    if err: print(f"   stderr:\n{err}")
    return "failed"


# ===== Modo automático: cola coordinada por Google Sheet =====
!pip install -q --upgrade gspread
import random
from datetime import datetime, timezone
import numpy as np
from IPython.display import Audio, display

# >>> EDITABLES <<<
SHEET_ID  = "10G9lkcjEY_Ns63zWXDTWIo_xT3th044QSYIdyOAqf0M"
SHEET_TAB = "Hoja 1"
WORKER    = ""     # opcional: nombre de ESTA cuenta. Vacío = autogenerado.
TTL_MIN   = 30     # fila DOING más vieja que esto = huérfana (reclamable)
VERIFY_S  = 3      # espera entre marcar DOING y releer para confirmar el claim
IDLE_S    = 60     # si no hay nada para mí, espero esto y reintento

# --- Auth + conexión al Sheet (popup 1 vez por cuenta) ---
from google.colab import auth
auth.authenticate_user()
import gspread
from gspread import Cell
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
ws = gc.open_by_key(SHEET_ID).worksheet(SHEET_TAB)

if not WORKER:
    import uuid; WORKER = "colab-" + uuid.uuid4().hex[:6]
print(f"Worker: {WORKER}  |  Sheet: {SHEET_ID[:10]}...  |  Tab: {SHEET_TAB}")

COL_STATUS, COL_WORKER, COL_TIME = 2, 3, 4   # B, C, D
local_stems = {p.stem for p in inputs}

def _now():  return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
def _stem(name):
    name = name.strip()
    for e in VIDEO_EXTS:
        if name.lower().endswith(e): return name[:-len(e)]
    return name
def _orphan(ctime):
    try:
        age = (datetime.now(timezone.utc) -
               datetime.strptime(ctime, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
              ).total_seconds() / 60
        return age > TTL_MIN
    except Exception:
        return True   # sin timestamp legible -> tratar como huérfana

done = skipped = failed = 0
t_global = time.time()

while True:
    rows = ws.get_all_values()[1:]           # 1 request: toda la tabla (sin encabezado)
    target = None
    for i, row in enumerate(rows, start=2):  # i = nro de fila real en el Sheet
        video  = row[0] if len(row) > 0 else ""
        status = (row[1] if len(row) > 1 else "").strip().upper()
        ctime  = row[3] if len(row) > 3 else ""
        if not video:                          continue
        if _stem(video) not in local_stems:    continue   # no subido a esta tanda -> otra cuenta
        if status in ("DONE", "FAILED"):       continue
        if status == "DOING" and not _orphan(ctime): continue   # tomado y vigente
        target = (i, _stem(video), video)
        break

    if target is None:
        pend = sum(1 for r in rows if r and r[0]
                   and (r[1] if len(r) > 1 else "").strip().upper() not in ("DONE", "FAILED"))
        if pend == 0:
            print("\nTodo DONE/FAILED. No queda nada por hacer."); break
        print(f"\nNada disponible para mí ahora ({pend} pendientes, tomadas por otras o no subidas)."
              f" Espero {IDLE_S}s..."); time.sleep(IDLE_S); continue

    rownum, stem, video = target

    # Ya hecho de antes (SRT presente): marco DONE y sigo, no reproceso.
    if (OUTPUT_DIR / f"{stem}.srt").exists():
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE")]); skipped += 1; continue

    # Claim: DOING + worker + timestamp (1 request).
    ws.update_cells([Cell(rownum, 2, "DOING"), Cell(rownum, 3, WORKER), Cell(rownum, 4, _now())])
    time.sleep(VERIFY_S)
    # Verify: releo la fila del servidor. Si la col 'worker' no soy yo, perdí la carrera.
    check = ws.row_values(rownum)
    if len(check) < 3 or check[2] != WORKER:
        continue

    # Gané: transcribo.
    print(f"\n[fila {rownum}] Procesando: {video}")
    vid_path = next((p for p in inputs if p.stem == stem), None)
    if vid_path is None:                       # se borró entre el scan y el claim
        ws.update_cells([Cell(rownum, 2, "TODO"), Cell(rownum, 3, ""), Cell(rownum, 4, "")]); continue
    try:
        outcome = transcribe_one(vid_path, f"fila {rownum}")
    except Exception as ex:
        print(f"   x excepcion: {ex}"); outcome = "failed"

    if outcome in ("done", "skipped"):
        ws.update_cells([Cell(rownum, 2, "DONE"), Cell(rownum, 3, WORKER), Cell(rownum, 4, _now())])
        done += outcome == "done"; skipped += outcome == "skipped"
    else:
        ws.update_cells([Cell(rownum, 2, "FAILED"), Cell(rownum, 3, WORKER), Cell(rownum, 4, _now())])
        failed += 1

print(f"\n=== Fin ({WORKER}) ===")
print(f"  transcritos: {done}  |  saltados: {skipped}  |  fallidos: {failed}")
print(f"  tiempo: {(time.time()-t_global)/60:.1f} min")

sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))


## 2) Setup (modo manual por rango)

Para el modo manual. Corré esta celda: instala el binario, monta Drive, lista `Host Videos`, te muestra el total + un reparto sugerido, y te da un cuadro para escribir el rango (ej. `1-100`). Después pasá a la celda 3.

> Si vas a usar el **modo automático (celda 1)**, ignorá las celdas 2 y 3.


In [ ]:
!apt-get -qq install -y ffmpeg p7zip-full > /dev/null

import os, time, re, subprocess, shutil
from pathlib import Path
from google.colab import drive
import torch

# --- Descargar + extraer faster-whisper-xxl (una sola vez por sesion) ---
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"

if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, \
            f"Descarga fallo o quedo incompleta (size={ARCHIVE.stat().st_size if ARCHIVE.exists() else 0})"
    print("Extrayendo (puede tardar 1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), f"Extraccion fallo - no aparecio {EXE}"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)

# --- Mount Drive ---
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# --- Carpetas (editar si tus nombres son distintos) ---
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: 'float16' y la extraccion de voz en 'cuda' van a fallar. Activa T4 GPU.")

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    # Orden "humano": los tramos de digitos se comparan como enteros,
    # asi 2 < 10 < 100 (no lexicografico, donde "100" caeria antes que "2").
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS),
                key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"

total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
pending = total - already
print(f"\n{total} archivo(s) en '{INPUT_DIR.name}'  |  {already} con SRT  |  {pending} pendientes")
print("Primeros 5:")
for i, p in enumerate(inputs[:5], 1):
    mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
    print(f"  {i:>4}. [{mark}] {p.name}")
if total > 10:
    print("  ...")
    for i, p in enumerate(inputs[-3:], total - 2):
        mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
        print(f"  {i:>4}. [{mark}] {p.name}")

# --- Carpeta temporal + armador del comando (flags en UN solo lugar) ---
TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

def build_cmd(vid):
    return [
        str(EXE), str(vid),
        # --- Modelo / idioma ---
        "--model", "large-v2",
        "--language", "ru",
        "--task", "transcribe",
        # --- Decoder ---
        "--initial_prompt", "None",
        "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "float16",
        "--temperature", "0",
        "--beam_size", "5",
        "--vad_filter", "True",
        # --- Audio: limpieza para musica / ruido / multi-hablante ---
        "--ff_vocal_extract", "mdx_kim2",
        "--voc_device", "cuda",
        "--ff_loudnorm",
        # --- Diarizacion: descomenta SOLO si necesitas etiquetas de hablante (pasada extra pesada) ---
        # "--diarize", "pyannote_v3.1",
        # "--diarize_device", "cuda",
        # --- Salida ---
        "--max_line_width", "200",
        "--max_line_count", "1",
        "--sentence",
        "--output_dir", str(TMP_OUT),
        "--output_format", "srt",
    ]

def transcribe_one(vid, label):
    """Corre el CLI para un video y mueve el SRT a Drive (sin pisar uno ya hecho).
    Devuelve 'done' | 'skipped' | 'failed'."""
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except OSError: pass
    t0 = time.time()
    result = subprocess.run(build_cmd(vid), capture_output=True, text=True)
    cli_srt = TMP_OUT / f"{vid.stem}.srt"
    if result.returncode == 0 and cli_srt.exists():
        if final_srt.exists():                       # otro la dejo mientras transcribia
            cli_srt.unlink(missing_ok=True)
            print(f"   = {label}: ya estaba hecho por otra cuenta; descarto el mio")
            return "skipped"
        shutil.move(str(cli_srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"
    print(f"   x {label}: CLI exit {result.returncode}; sin SRT.")
    err = (result.stderr or "")[-400:]
    if err: print(f"   stderr:\n{err}")
    return "failed"

import ipywidgets as widgets
from IPython.display import display

# --- Reparto sugerido entre 5 cuentas ---
N_ACCOUNTS = 5
chunk = (total + N_ACCOUNTS - 1) // N_ACCOUNTS
print(f"\nReparto sugerido para {N_ACCOUNTS} cuentas (~{chunk} videos cada una):")
for k in range(N_ACCOUNTS):
    a = k * chunk + 1
    b = min((k + 1) * chunk, total)
    if a > total: break
    print(f"   cuenta {k+1} -> {a}-{b}")

# --- Cuadro de input para el rango ---
range_box = widgets.Text(
    value=f"1-{min(100, total)}",
    placeholder="ej: 1-100  (o 'all' para todo)",
    description="Rango:",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "60px"},
)
print(f"\nElegi el rango (1..{total}) y pasa a la celda 3:")
display(range_box)


## 3) Transcribir un rango (modo manual)

Lee el rango del cuadro de la celda 2 y transcribe solo esos videos. Salta los que ya tienen SRT en Drive.


In [ ]:
import numpy as np
from IPython.display import Audio, display

# --- Parsear el rango del cuadro de la celda 2 ---
assert "range_box" in globals(), "Primero corre la celda 2 (setup manual)."
spec = (range_box.value or "").strip().lower()
if spec in ("", "all"):
    start, end = 1, total
else:
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", spec) or re.fullmatch(r"(\d+)", spec)
    assert m, f"Rango invalido: {spec!r}. Usa '1-100' o '50' o 'all'."
    if m.lastindex == 1:
        start = end = int(m.group(1))
    else:
        start, end = int(m.group(1)), int(m.group(2))
start = max(1, start); end = min(total, end)
assert start <= end, f"Rango vacio despues de recortar a 1..{total}: {start}-{end}"

batch = inputs[start-1:end]
print(f"Procesando {len(batch)} archivo(s): #{start} a #{end} de {total}.")

done = skipped = failed = 0
t_global = time.time()
for offset, vid in enumerate(batch):
    i = start + offset
    if (OUTPUT_DIR / f"{vid.stem}.srt").exists():
        print(f"\n[{i}/{end}] SALTADO (ya existe en Drive): {vid.stem}.srt")
        skipped += 1
        continue
    print(f"\n[{i}/{end}] Procesando: {vid.name}")
    try:
        outcome = transcribe_one(vid, f"{i}/{end}")
    except Exception as ex:
        print(f"   x excepcion: {ex}"); outcome = "failed"
    done    += outcome == "done"
    skipped += outcome == "skipped"
    failed  += outcome == "failed"

print(f"\n=== Resumen del rango {start}-{end} ===")
print(f"  procesados: {done}")
print(f"  saltados:   {skipped}")
print(f"  fallidos:   {failed}")
print(f"  tiempo total: {(time.time()-t_global)/60:.1f} min")
print(f"\nSRT en: {OUTPUT_DIR}")

# Ruidito final
sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
